# 02 — Data Cleaning

This notebook documents all data cleaning steps performed on the 10 raw datasets.

Cleaning is executed via the ETL scripts:
- `scripts/clean_nav.py` — NAV history cleaning
- `scripts/clean_transactions.py` — Investor transactions
- `scripts/clean_performance.py` — Scheme performance
- `scripts/copy_remaining.py` — Copies remaining CSVs

Run the full pipeline: `python scripts/etl_pipeline.py`

## Steps Applied
1. **Date parsing** — `pd.to_datetime(errors="coerce")`; unparseable rows dropped
2. **NAV validation** — nav > 0 enforced; zero/negative dropped
3. **Forward-fill** — `ffill()` after reindexing to full trading-day calendar (handles weekends/holidays)
4. **Deduplication** — (amfi_code, date) composite key enforced
5. **Amount normalisation** — Investor transaction amounts standardised to INR
6. **Foreign key integrity** — Orphan amfi_codes removed before DB load

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path().resolve().parent
PROCESSED = ROOT / 'data' / 'processed'

for f in sorted(PROCESSED.glob('*.csv')):
    df = pd.read_csv(f)
    print(f'{f.name:40s}  {len(df):>6,} rows  {len(df.columns):>3} cols')

In [ ]:
# NAV Cleaning Summary
nav = pd.read_csv(PROCESSED / 'clean_nav.csv', parse_dates=['date'])
print('NAV date range:', nav['date'].min(), '→', nav['date'].max())
print('Funds:', nav['amfi_code'].nunique())
print('Total records:', len(nav))
print('Missing NAV values:', nav['nav'].isna().sum())
print(nav.dtypes)

In [ ]:
# Transaction Cleaning Summary
txn = pd.read_csv(PROCESSED / 'clean_transactions.csv', parse_dates=['transaction_date'])
print('Transaction types:', txn['transaction_type'].value_counts().to_dict())
print('Date range:', txn['transaction_date'].min(), '→', txn['transaction_date'].max())
print('Missing values:'); print(txn.isna().sum())